# EU_06 – Navigation of Mobile Robots
> **Robotics Frameworks (RoF) · FAU Erlangen-Nürnberg · WS 2025/26**

---
> ⚠️ **Note:** SLAM Toolbox and Nav2 require ROS2 + Gazebo (run on your VM).
> This notebook covers the **theory and algorithms** behind navigation in runnable Python.

## What this notebook covers
| Part | Topic |
|------|-------|
| **1** | Occupancy Grid Maps |
| **2** | Path Planning: BFS, Dijkstra, A* |
| **3** | 1D Kalman Filter (sensor fusion) |
| **4** | SLAM Concept Simulation |
| **5** | ROS2 Nav2 Architecture (reference) |

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import heapq
from collections import deque

print('All libraries ready!')

---
## Part 1: Occupancy Grid Maps

An **occupancy grid** divides the environment into a regular 2D grid.
Each cell stores the probability that it is occupied by an obstacle:
- `0` = free space (white)
- `1` = occupied / obstacle (black)
- `0.5` = unknown (grey)

This is the map representation used by **slam_toolbox** and **Nav2 costmap**.
Resolution in our lab configuration: **5 cm per cell** (`resolution: 0.05` in `slam_params.yaml`).

In [ ]:
# ── Build a simple occupancy grid (FAPS lab-like layout) ─────────────────────
GRID_SIZE = 20

grid = np.zeros((GRID_SIZE, GRID_SIZE))  # 0 = free

# Add walls (outer border)
grid[0, :]  = 1
grid[-1, :] = 1
grid[:, 0]  = 1
grid[:, -1] = 1

# Add internal obstacles (shelves / walls in FAPS lab)
grid[4, 2:8]   = 1   # horizontal wall
grid[4:12, 8]  = 1   # vertical wall
grid[12, 8:16] = 1   # horizontal wall
grid[8, 12:18] = 1   # another wall

# Start and Goal positions
START = (2, 2)
GOAL  = (17, 17)

def plot_grid(grid, path=None, visited=None, title='Occupancy Grid'):
    fig, ax = plt.subplots(figsize=(7, 7))
    display = grid.copy().astype(float)

    if visited:
        for r, c in visited:
            if (r, c) != START and (r, c) != GOAL:
                display[r, c] = 0.6  # light blue for explored

    if path:
        for r, c in path:
            if (r, c) != START and (r, c) != GOAL:
                display[r, c] = 0.3  # green-ish for path

    ax.imshow(display, cmap='RdYlGn_r', vmin=0, vmax=1,
              origin='upper', interpolation='none')

    sr, sc = START
    gr, gc = GOAL
    ax.plot(sc, sr, 'bs', markersize=14, label='Start')
    ax.plot(gc, gr, 'r*', markersize=16, label='Goal')

    if path and len(path) > 1:
        pr = [p[0] for p in path]
        pc = [p[1] for p in path]
        ax.plot(pc, pr, 'b-', linewidth=2.5, label=f'Path ({len(path)} cells)')

    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

plot_grid(grid, title='Occupancy Grid — FAPS-style Environment')

---
## Part 2: Path Planning Algorithms

Nav2 uses **NavFn (Dijkstra / A*)** as its global planner. Here we implement
all three algorithms from scratch to compare their behaviour.

| Algorithm | Strategy | Optimal? | Complete? | Uses Heuristic? |
|-----------|----------|----------|-----------|----------------|
| **BFS** | Breadth-first (layer by layer) | ✅ (uniform cost) | ✅ | ❌ |
| **Dijkstra** | Lowest cost first | ✅ | ✅ | ❌ |
| **A*** | Lowest (cost + heuristic) first | ✅ | ✅ | ✅ (Manhattan / Euclidean) |

In [ ]:
# ── Shared helpers ────────────────────────────────────────────────────────────
def get_neighbors(pos, grid):
    r, c = pos
    rows, cols = grid.shape
    # 4-connected (up, down, left, right) — Nav2 supports 8-connected too
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr, nc = r+dr, c+dc
        if 0 <= nr < rows and 0 <= nc < cols and grid[nr, nc] == 0:
            yield (nr, nc)

def reconstruct_path(came_from, start, goal):
    path, node = [], goal
    while node != start:
        path.append(node)
        node = came_from[node]
    path.append(start)
    path.reverse()
    return path


# ── 2.1 BFS ──────────────────────────────────────────────────────────────────
def bfs(grid, start, goal):
    queue = deque([start])
    came_from = {start: None}
    visited = set()

    while queue:
        current = queue.popleft()
        visited.add(current)
        if current == goal:
            return reconstruct_path(came_from, start, goal), visited
        for nb in get_neighbors(current, grid):
            if nb not in came_from:
                came_from[nb] = current
                queue.append(nb)
    return [], visited


# ── 2.2 Dijkstra ─────────────────────────────────────────────────────────────
def dijkstra(grid, start, goal):
    pq = [(0, start)]
    came_from = {start: None}
    cost_so_far = {start: 0}
    visited = set()

    while pq:
        cost, current = heapq.heappop(pq)
        visited.add(current)
        if current == goal:
            return reconstruct_path(came_from, start, goal), visited
        for nb in get_neighbors(current, grid):
            new_cost = cost_so_far[current] + 1
            if nb not in cost_so_far or new_cost < cost_so_far[nb]:
                cost_so_far[nb] = new_cost
                came_from[nb] = current
                heapq.heappush(pq, (new_cost, nb))
    return [], visited


# ── 2.3 A* ───────────────────────────────────────────────────────────────────
def heuristic(a, b):
    # Manhattan distance (used for 4-connected grids)
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def astar(grid, start, goal):
    pq = [(0, start)]
    came_from = {start: None}
    g_score = {start: 0}   # actual cost from start
    visited = set()

    while pq:
        f, current = heapq.heappop(pq)
        visited.add(current)
        if current == goal:
            return reconstruct_path(came_from, start, goal), visited
        for nb in get_neighbors(current, grid):
            new_g = g_score[current] + 1
            if nb not in g_score or new_g < g_score[nb]:
                g_score[nb] = new_g
                f_score = new_g + heuristic(nb, goal)  # f = g + h
                came_from[nb] = current
                heapq.heappush(pq, (f_score, nb))
    return [], visited


# ── Run all three and compare ─────────────────────────────────────────────────
path_bfs,  vis_bfs  = bfs(grid, START, GOAL)
path_dijk, vis_dijk = dijkstra(grid, START, GOAL)
path_astar,vis_astar= astar(grid, START, GOAL)

for name, path, vis in [
    (f'BFS         — Path: {len(path_bfs)} cells | Explored: {len(vis_bfs)} cells', path_bfs, vis_bfs),
    (f'Dijkstra    — Path: {len(path_dijk)} cells | Explored: {len(vis_dijk)} cells', path_dijk, vis_dijk),
    (f'A* (Manhattan) — Path: {len(path_astar)} cells | Explored: {len(vis_astar)} cells', path_astar, vis_astar),
]:
    plot_grid(grid, path=path, visited=vis, title=name)

In [ ]:
# ── Summary comparison bar chart ──────────────────────────────────────────────
algorithms = ['BFS', 'Dijkstra', 'A*']
paths      = [len(path_bfs), len(path_dijk), len(path_astar)]
explored   = [len(vis_bfs), len(vis_dijk), len(vis_astar)]

x = np.arange(3)
fig, ax = plt.subplots(figsize=(8, 4))
bars1 = ax.bar(x - 0.2, explored, 0.35, label='Cells Explored', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + 0.2, paths,    0.35, label='Path Length',    color='#2ecc71', alpha=0.8)

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(int(bar.get_height())), ha='center', va='bottom', fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(algorithms, fontsize=12)
ax.set_ylabel('Number of Cells')
ax.set_title('Path Planning Comparison — Efficiency', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Key insight: A* explores FEWER cells than BFS/Dijkstra')
print('because the heuristic guides the search toward the goal.')
print(f'A* efficiency gain: {len(vis_bfs)-len(vis_astar)} fewer cells explored vs BFS')

---
## Part 3: 1D Kalman Filter — Sensor Fusion for Localization

The Kalman Filter is used in robot localization (e.g., combining **wheel odometry + GPS/LIDAR**).
It optimally estimates the true state from **noisy measurements**.

### The two steps:
```
PREDICT:  x̂⁻ = A·x̂ + B·u       (use motion model)
          P⁻  = A·P·Aᵀ + Q       (increase uncertainty)

UPDATE:   K   = P⁻·Hᵀ / (H·P⁻·Hᵀ + R)   (Kalman gain)
          x̂  = x̂⁻ + K·(z - H·x̂⁻)       (correct with measurement)
          P   = (I - K·H)·P⁻              (decrease uncertainty)
```
where:
- `x̂` = state estimate (position), `P` = uncertainty
- `Q` = process noise (how much does motion model drift?)
- `R` = measurement noise (how noisy is the sensor?)

In [ ]:
# ── 1D Kalman Filter: robot driving in a straight line ────────────────────────
np.random.seed(42)

N     = 50           # number of time steps
dt    = 0.1          # time step (seconds)
v     = 1.0          # constant velocity (m/s)
Q     = 0.01         # process noise variance (odometry drift)
R     = 0.5          # measurement noise variance (GPS/LIDAR noise)

# ── Ground truth ──────────────────────────────────────────────────────────────
true_positions = np.array([v * dt * t for t in range(N)])

# ── Noisy odometry (wheel encoders — low noise, but drifts) ──────────────────
odom_positions = true_positions + np.random.normal(0, np.sqrt(Q)*10, N).cumsum()

# ── Noisy measurements (GPS/LIDAR — high noise, no drift) ────────────────────
measurements = true_positions + np.random.normal(0, np.sqrt(R), N)

# ── Kalman Filter ────────────────────────────────────────────────────────────
x_est  = 0.0    # initial estimate
P_est  = 1.0    # initial uncertainty
kf_estimates = []

for t in range(N):
    # PREDICT
    x_pred = x_est + v * dt     # motion model: move forward
    P_pred = P_est + Q           # uncertainty grows with motion

    # UPDATE with measurement
    K      = P_pred / (P_pred + R)              # Kalman gain
    x_est  = x_pred + K * (measurements[t] - x_pred)  # fuse prediction + measurement
    P_est  = (1 - K) * P_pred                   # reduce uncertainty

    kf_estimates.append(x_est)

kf_estimates = np.array(kf_estimates)

# ── Plot ─────────────────────────────────────────────────────────────────────
t_axis = np.arange(N) * dt

plt.figure(figsize=(12, 5))
plt.plot(t_axis, true_positions, 'k-',   linewidth=2.5, label='Ground Truth')
plt.plot(t_axis, measurements,   'r.',   markersize=6,  label=f'GPS/LIDAR (noise σ={np.sqrt(R):.2f}m)', alpha=0.6)
plt.plot(t_axis, odom_positions, 'b--',  linewidth=1.5, label='Odometry (drifts)', alpha=0.7)
plt.plot(t_axis, kf_estimates,   'g-',   linewidth=2.5, label='Kalman Filter Estimate')

plt.xlabel('Time (s)')
plt.ylabel('Position (m)')
plt.title('1D Kalman Filter — Fusing Odometry + GPS/LIDAR', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

rmse_meas = np.sqrt(np.mean((measurements - true_positions)**2))
rmse_kf   = np.sqrt(np.mean((kf_estimates - true_positions)**2))
print(f'RMSE (raw measurements): {rmse_meas:.4f} m')
print(f'RMSE (Kalman Filter):    {rmse_kf:.4f} m')
print(f'Improvement: {(1 - rmse_kf/rmse_meas)*100:.1f}% reduction in error')

---
## Part 4: SLAM Concept Simulation

**SLAM = Simultaneous Localization and Mapping**

The chicken-and-egg problem:
- To build a map, you need to know where you are
- To know where you are, you need a map

SLAM solves both **at the same time** using scan matching.

Here we simulate how `slam_toolbox` incrementally builds an occupancy grid
as the robot moves and takes LIDAR scans.

In [ ]:
# ── Simulated SLAM: robot moves, LIDAR builds map step by step ────────────────
MAP_SIZE = 25
slam_map = np.full((MAP_SIZE, MAP_SIZE), 0.5)  # 0.5 = unknown

# True obstacles (unknown to robot initially)
true_obstacles = set()
for c in range(5, 15):  true_obstacles.add((3, c))   # top wall
for r in range(3, 12):  true_obstacles.add((r, 15))  # right wall
for c in range(10, 20): true_obstacles.add((12, c))  # bottom wall

def lidar_scan(robot_pos, true_obs, max_range=8, n_rays=36):
    """Simulate a 2D LIDAR scan: returns set of (free, hit) cells."""
    rx, ry = robot_pos
    free_cells = set()
    hit_cells  = set()
    for i in range(n_rays):
        angle = 2 * np.pi * i / n_rays
        for d in np.arange(0.5, max_range, 0.5):
            cx = int(round(rx + d * np.cos(angle)))
            cy = int(round(ry + d * np.sin(angle)))
            if not (0 <= cx < MAP_SIZE and 0 <= cy < MAP_SIZE):
                break
            if (cy, cx) in true_obs:
                hit_cells.add((cy, cx))
                break
            else:
                free_cells.add((cy, cx))
    return free_cells, hit_cells

# Robot path (simulated exploration)
robot_path = [(7,2),(7,4),(7,6),(7,8),(7,10),(7,12),(9,12),(11,12),(11,10),(11,8)]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for step, robot_pos in enumerate(robot_path):
    free, hits = lidar_scan(robot_pos, true_obstacles, max_range=7)
    for r, c in free:  slam_map[r, c] = max(slam_map[r, c] - 0.15, 0.0)
    for r, c in hits:  slam_map[r, c] = min(slam_map[r, c] + 0.4,  1.0)

    disp = slam_map.copy()
    ax = axes[step]
    ax.imshow(disp, cmap='RdYlGn_r', vmin=0, vmax=1, origin='upper', interpolation='none')
    ry, rx = robot_pos
    ax.plot(rx, ry, 'b^', markersize=10)
    # Draw path so far
    if step > 0:
        pr = [p[0] for p in robot_path[:step+1]]
        pc = [p[1] for p in robot_path[:step+1]]
        ax.plot(pc, pr, 'b-', linewidth=1, alpha=0.5)
    ax.set_title(f'Step {step+1}: pos=({rx},{ry})', fontsize=9)
    ax.axis('off')

plt.suptitle('SLAM Incremental Map Building (white=free, black=obstacle, grey=unknown)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('Blue triangle = robot | As robot explores, unknown (grey) cells become known')

---
## Part 5: ROS2 Nav2 Architecture Reference

This is what runs on your **VirtualBox VM** — cannot run in Colab (needs ROS2).
Use this as a reference for your exam.

In [ ]:
# ── Nav2 Architecture Diagram ─────────────────────────────────────────────────
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14); ax.set_ylim(0, 7); ax.axis('off')
ax.set_title('Nav2 Stack Architecture (EU_06)', fontsize=14, fontweight='bold', pad=15)

components = [
    # (x, y, w, h, label, color)
    (0.3, 4.5, 2.5, 1.2, 'LIDAR\n/scan', '#3498db'),
    (0.3, 2.5, 2.5, 1.2, 'Odometry\n/odom', '#3498db'),
    (0.3, 0.5, 2.5, 1.2, 'Gazebo /\nReal Robot', '#95a5a6'),
    (3.5, 4.5, 2.5, 1.2, 'slam_toolbox\n(SLAM)', '#2ecc71'),
    (3.5, 2.8, 2.5, 1.2, 'nav2_amcl\n(Localization)', '#2ecc71'),
    (3.5, 1.0, 2.5, 1.2, 'nav2_map_server\n(Load saved map)', '#2ecc71'),
    (7.0, 4.5, 2.5, 1.2, 'nav2_planner\n(A* Global Path)', '#e67e22'),
    (7.0, 2.8, 2.5, 1.2, 'nav2_costmap_2d\n(Inflate obstacles)', '#e67e22'),
    (7.0, 1.0, 2.5, 1.2, 'nav2_controller\n(DWB Local)', '#e67e22'),
    (10.5, 3.0, 2.5, 1.5, 'nav2_bt_navigator\n(Behavior Tree\nOrchestrator)', '#9b59b6'),
    (10.5, 0.5, 2.5, 1.2, '/cmd_vel\n(Robot motion)', '#e74c3c'),
]

boxes = {}
for x, y, w, h, label, color in components:
    rect = mpatches.FancyBboxPatch((x, y), w, h,
        boxstyle='round,pad=0.1', facecolor=color, alpha=0.85,
        edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x+w/2, y+h/2, label, ha='center', va='center',
            fontsize=8, fontweight='bold', color='white')
    boxes[label.split('\n')[0]] = (x+w, y+h/2, x, y+h/2)  # right, left centers

# Arrows
arrow_kw = dict(arrowstyle='->', color='#2c3e50', lw=1.5)
arrows = [
    ((2.8,5.1),(3.5,5.1)),  # LIDAR → SLAM
    ((2.8,3.1),(3.5,3.4)),  # Odom → AMCL
    ((6.0,5.1),(7.0,5.1)),  # SLAM → Planner
    ((6.0,3.4),(7.0,3.4)),  # AMCL → Costmap
    ((6.0,1.6),(7.0,1.6)),  # Map Server → Controller
    ((9.5,5.1),(10.5,3.8)), # Planner → BT Nav
    ((9.5,1.6),(10.5,3.0)), # Controller → BT Nav
    ((11.75,3.0),(11.75,1.7)),# BT Nav → cmd_vel
]
for (x1,y1),(x2,y2) in arrows:
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=arrow_kw)

# TF Chain label
ax.text(7, 6.5,
    'TF Chain:  map → odom → base_footprint → base_link → lidar_link',
    ha='center', fontsize=9, style='italic',
    bbox=dict(boxstyle='round', facecolor='#eaf2ff', alpha=0.9))

# Legend
legend_els = [
    mpatches.Patch(color='#3498db', label='Sensors / Robot HW'),
    mpatches.Patch(color='#2ecc71', label='Localization / SLAM'),
    mpatches.Patch(color='#e67e22', label='Planning & Control'),
    mpatches.Patch(color='#9b59b6', label='Orchestration (BT)'),
]
ax.legend(handles=legend_els, loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

---
## Part 6: EU_06 Exercise Workflow (VM Reference)

This cannot run in Colab — use your **VirtualBox VM** for this.

### Phase 1: Build the Map (SLAM Mode)
```bash
# Terminal 1 — Gazebo simulation
ros2 launch rof_gazebo t3_simulation_faps.launch.py

# Terminal 2 — SLAM (builds map from /scan)
ros2 launch rof_ex6 slam_launch.py

# Terminal 3 — Visualise
ros2 launch nav2_bringup rviz_launch.py

# Terminal 4 — Drive manually to explore
ros2 run rqt_robot_steering rqt_robot_steering
```

### Phase 2: Save the Map
```bash
# In rqt → Plugins → Services → Service Caller
# Service: /slam_toolbox/serialize_map
# filename: '/home/robot/map'
```

### Phase 3: Autonomous Navigation
```bash
# Edit slam_params.yaml:
#   mode: localization
#   map_file_name: /home/robot/map

ros2 launch rof_ex6 slam_launch.py       # localization mode
ros2 launch rof_ex6 navigation_launch.py  # full Nav2 stack
ros2 launch nav2_bringup rviz_launch.py   # RViz — click 2D Nav Goal!
```

---
## Summary

| Concept | What it does | Where it runs |
|---------|-------------|---------------|
| Occupancy Grid | 2D map of free/occupied cells | slam_toolbox → /map |
| A* | Finds shortest path avoiding obstacles | nav2_planner |
| Kalman Filter | Fuses odometry + sensor data | nav2_amcl (particle filter variant) |
| SLAM | Builds map while localizing | slam_toolbox |
| Nav2 | Full navigation stack | VM (ROS2 Humble) |